# Train the password strength model on Kaggle

This private notebook prepares the project's password datasets, trains Random Forest and XGBoost, evaluates them, and packages model artifacts for the local app. It checkpoints outputs under `/kaggle/working/password_manager_training/checkpoints/` after each stage.

> **Keep this notebook and its attached datasets private.** The source data contains leaked/common passwords. Never add real personal passwords. Do not publish prepared CSVs, dictionary files, or raw datasets as notebook outputs. The final download zip contains model artifacts, metadata, and evaluation output only.

## Before running
1. Create a **private Kaggle Dataset** containing `prepare_data.py`, `features.py`, `train_model.py`, `evaluate_model.py`, and these input files with their exact names: `password_strength.csv`, `rockyou.txt`, `xato-net-10-million-passwords.csv`, and `top10k-common-passwords.csv`. The Python files are in this repository's `proposed-improvement/ml-model/`; the datasets are in `datasets/`. Obtain RockYou from the repository's Git LFS download or another authorized source. The Git LFS pointer text is not the dataset.
2. Create a Kaggle Notebook from this `.ipynb`, attach that private dataset as an input, and keep the notebook private.
3. Enable Internet in Notebook settings if Kaggle needs to install packages. Run the cells in order.
4. After a run completes, use **Save Version → Save & Run All** to preserve Kaggle notebook outputs. Download `password-manager-kaggle-model.zip` from the notebook output files. Intermediate `/kaggle/working` checkpoints are for the active run; they are not a substitute for saving a Kaggle version.

Labels are partly source-derived and heuristic. Evaluation scores measure agreement with those labels, not measured crack resistance.

In [ ]:
%pip install -q pandas numpy scikit-learn xgboost joblib matplotlib seaborn zxcvbn-python

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, zipfile
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
RUN_DIR = Path('/kaggle/working/password_manager_training')
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
DATA_DIR = RUN_DIR / 'datasets'
MODEL_DIR = RUN_DIR / 'models'
FIGURE_DIR = RUN_DIR / 'figures'
for folder in (CHECKPOINT_DIR, DATA_DIR, MODEL_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def find_input(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Missing {filename}. Attach the private assets dataset described above.')
    return matches[0]

CODE_DIR = find_input('prepare_data.py').parent
required_files = {name: find_input(name) for name in [
    'password_strength.csv', 'rockyou.txt',
    'xato-net-10-million-passwords.csv', 'top10k-common-passwords.csv'
]}
rockyou_head = required_files['rockyou.txt'].read_bytes()[:100]
if b'git-lfs.github.com/spec' in rockyou_head:
    raise RuntimeError('rockyou.txt is only a Git LFS pointer. Download the real LFS file and re-upload it to the private Kaggle Dataset.')
print('Code:', CODE_DIR)
print('Inputs:', {key: str(value) for key, value in required_files.items()})
print('Run directory:', RUN_DIR)

## Checkpoint 1 — input inventory
This saves paths and sizes only, not password contents. Confirm the RockYou file is real data rather than an LFS pointer before proceeding.

In [ ]:
inventory = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'code_dir': str(CODE_DIR),
    'inputs': {name: {'path': str(path), 'bytes': path.stat().st_size} for name, path in required_files.items()},
}
(CHECKPOINT_DIR / '01_input_inventory.json').write_text(json.dumps(inventory, indent=2))
print(json.dumps(inventory, indent=2))

## Checkpoint 2 — prepare data
The script writes a training CSV, a held-out multi-source CSV, and a dictionary into the Kaggle working directory. These files contain password strings: keep them private and do not publish them as outputs.

In [ ]:
prepare_cmd = [sys.executable, str(CODE_DIR / 'prepare_data.py'),
    '--kaggle-path', str(required_files['password_strength.csv']),
    '--rockyou-path', str(required_files['rockyou.txt']),
    '--xato-path', str(required_files['xato-net-10-million-passwords.csv']),
    '--top10k-path', str(required_files['top10k-common-passwords.csv']),
    '--seclists-path', str(INPUT_ROOT / 'seclists'),
    '--output-dir', str(DATA_DIR), '--rockyou-limit', '200000',
    '--xato-limit', '500000', '--n-strong', '50000']
prepare_log = CHECKPOINT_DIR / '02_prepare.log'
with prepare_log.open('w', encoding='utf-8') as log:
    subprocess.run(prepare_cmd, cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT, check=True)
print(prepare_log.read_text(encoding='utf-8')[-12000:])
train_csv = DATA_DIR / 'combined_train.csv'
test_csv = DATA_DIR / 'clean_test.csv'
dict_txt = DATA_DIR / 'dictionary.txt'
train_frame = pd.read_csv(train_csv)
test_frame = pd.read_csv(test_csv)
prepared_summary = {
    'train_rows': int(len(train_frame)),
    'train_source_counts': {str(k): int(v) for k, v in train_frame['source'].value_counts().items()},
    'train_label_counts': {str(int(k)): int(v) for k, v in train_frame['strength'].value_counts().sort_index().items()},
    'test_rows': int(len(test_frame)),
    'test_source_counts': {str(k): int(v) for k, v in test_frame['source'].value_counts().items()},
    'test_label_counts': {str(int(k)): int(v) for k, v in test_frame['strength'].value_counts().sort_index().items()},
    'dictionary_entries': sum(1 for _ in dict_txt.open(encoding='utf-8', errors='ignore')),
}
(CHECKPOINT_DIR / '02_prepared_data_summary.json').write_text(json.dumps(prepared_summary, indent=2))
print(json.dumps(prepared_summary, indent=2))
del train_frame, test_frame

## Checkpoint 3 — train and save models
The updated training script atomically saves `feature_names.joblib` before model fitting, then saves `rf_model.joblib` immediately after Random Forest training and `xgb_model.joblib` immediately after XGBoost training. If XGBoost fails, the completed Random Forest artifact remains in `/kaggle/working`.

In [ ]:
train_cmd = [sys.executable, str(CODE_DIR / 'train_model.py'),
    '--dataset-path', str(train_csv), '--dict-path', str(dict_txt),
    '--output-dir', str(MODEL_DIR)]
train_log = CHECKPOINT_DIR / '03_train.log'
with train_log.open('w', encoding='utf-8') as log:
    subprocess.run(train_cmd, cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT, check=True)
print(train_log.read_text(encoding='utf-8')[-16000:])
required_artifacts = ['feature_names.joblib', 'rf_model.joblib', 'xgb_model.joblib']
missing = [name for name in required_artifacts if not (MODEL_DIR / name).is_file()]
if missing: raise FileNotFoundError(f'Missing trained artifacts: {missing}')
(CHECKPOINT_DIR / '03_models_checkpoint.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'artifacts': {name: (MODEL_DIR / name).stat().st_size for name in required_artifacts}
}, indent=2))
print('Model checkpoints saved:', sorted(p.name for p in MODEL_DIR.iterdir()))

## Checkpoint 4 — evaluate
Evaluation uses the regenerated training and clean test CSVs. The clean test labels are source-derived/heuristic proxies; describe this limitation when reporting scores.

In [ ]:
eval_cmd = [sys.executable, str(CODE_DIR / 'evaluate_model.py'),
    '--dataset-path', str(train_csv), '--clean-test-path', str(test_csv),
    '--dict-path', str(dict_txt), '--models-dir', str(MODEL_DIR),
    '--figures-dir', str(FIGURE_DIR)]
eval_log = CHECKPOINT_DIR / '04_evaluation.log'
with eval_log.open('w', encoding='utf-8') as log:
    subprocess.run(eval_cmd, cwd=CODE_DIR, stdout=log, stderr=subprocess.STDOUT, check=True)
print(eval_log.read_text(encoding='utf-8')[-20000:])
(CHECKPOINT_DIR / '04_evaluation_checkpoint.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'figures': sorted(p.name for p in FIGURE_DIR.glob('*.png'))
}, indent=2))

## Checkpoint 5 — package artifacts for the app
The zip intentionally excludes prepared CSVs and the dictionary. Keep the output private. Download it from the notebook output panel and provide it for integration into the app.

In [ ]:
bundle = Path('/kaggle/working/password-manager-kaggle-model.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ['xgb_model.joblib', 'feature_names.joblib', 'training_metadata.json']:
        zf.write(MODEL_DIR / name, arcname=f'models/{name}')
    for name in ['04_evaluation.log']:
        zf.write(CHECKPOINT_DIR / name, arcname=f'results/{name}')
    for image in FIGURE_DIR.glob('*.png'):
        zf.write(image, arcname=f'results/figures/{image.name}')
    zf.writestr('README.txt', 'Private Kaggle-trained XGBoost artifact. Copy xgb_model.joblib and feature_names.joblib into proposed-improvement/ml-model/models/. Keep this bundle private. Check the XGBoost and scikit-learn versions recorded in your Kaggle notebook before loading with joblib.')
(CHECKPOINT_DIR / '05_package_checkpoint.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(), 'bundle': str(bundle), 'bytes': bundle.stat().st_size
}, indent=2))
print('Download this private model bundle:', bundle)
from IPython.display import FileLink, display
display(FileLink(str(bundle)))

## What to send back for integration
Download `password-manager-kaggle-model.zip` and attach it in this chat. I will check the model features and library versions, update the local API's XGBoost artifact and feature-name file, run the API prediction check, rebuild the frontend if any code changes are needed, and prepare the final Git update. The UI already calls `/ml-api/predict`; it does not need to contain the model itself.